# Avito AntiBot

Фиксированное решение с temporal validation и ранговым ансамблем LightGBM. Признаки, модели, параметры, split и метрика перенесены из исходного ноутбука без изменений.

Запускайте ячейки сверху вниз. Если ноутбук открыт отдельно, в разделе загрузки выберите ZIP проекта; пути вручную менять не нужно.


## 1. Setup

Закреплённые зависимости устанавливаются один раз до импорта библиотек.


In [ ]:
import sys
import subprocess
from importlib.metadata import PackageNotFoundError, version

PINS = {"numpy":"2.3.5","pandas":"2.2.3","scipy":"1.17.0","scikit-learn":"1.8.0","lightgbm":"4.7.0"}
missing = []
for package, expected in PINS.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    if installed != expected:
        missing.append(f"{package}=={expected}")
if missing:
    print("Устанавливаю зависимости:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Версии зависимостей:", PINS)


### Imports


In [ ]:
import importlib, json, shutil, sys, zipfile
from pathlib import Path, PurePosixPath
import numpy as np
import pandas as pd


## 2. Load data

Архив проекта содержит исходные CSV и Python-модули. В Colab загрузите ZIP один раз, если файлы ещё не находятся в текущей папке.


In [ ]:
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
VALIDATION_DIR = OUTPUT_DIR / "validation"
DATA_DIR.mkdir(parents=True, exist_ok=True)
VALIDATION_DIR.mkdir(parents=True, exist_ok=True)
MODULES = {"features.py", "metric.py", "pipeline.py", "submission.py"}
RAW = {"train.csv", "test.csv", "events.csv.gz", "events_part1.csv.gz", "events_part2.csv.gz", "events_part3.csv.gz"}

def extract_project_zip(path):
    with zipfile.ZipFile(path) as archive:
        for item in archive.infolist():
            parts = PurePosixPath(item.filename).parts
            if item.is_dir() or not parts or ".." in parts or PurePosixPath(item.filename).is_absolute():
                continue
            name = parts[-1]
            if name in MODULES:
                target = PROJECT_ROOT / name
            elif name in RAW:
                target = DATA_DIR / name
            else:
                continue
            if not target.exists():
                target.write_bytes(archive.read(item))

def missing_inputs():
    missing = [name for name in MODULES if not (PROJECT_ROOT / name).is_file()]
    for name in ("train.csv", "test.csv"):
        if not (DATA_DIR / name).is_file():
            missing.append("data/" + name)
    event_parts = all((DATA_DIR / f"events_part{i}.csv.gz").is_file() for i in (1,2,3))
    if not (DATA_DIR / "events.csv.gz").is_file() and not event_parts:
        missing.append("data/events.csv.gz или три events_part*.csv.gz")
    return missing

missing = missing_inputs()
if missing:
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError("Не найдены файлы: " + ", ".join(missing) + ". Распакуйте ZIP проекта в одну папку.") from exc
    print("Выберите ZIP проекта Avito_AntiBot_Colab.")
    files.upload()
    for archive_path in sorted(PROJECT_ROOT.glob("*.zip")):
        extract_project_zip(archive_path)
    missing = missing_inputs()
if missing:
    raise FileNotFoundError("Не найдены файлы: " + ", ".join(missing))

sys.path.insert(0, str(PROJECT_ROOT.resolve()))
importlib.invalidate_caches()
import pipeline as antibot_pipeline
from metric import precision_at_recall
assert precision_at_recall([1,0,1,1,1],[5,4,3,2,1]) == 0.8
assert precision_at_recall([1,1,0,0],[0.5]*4) == 0.5
assert precision_at_recall([0,0,1,1],[0.5]*4) == 0.5
train, test, events = antibot_pipeline.read_data(PROJECT_ROOT)
print("Train:", train.shape, "Test:", test.shape, "Events:", events.shape)


## 3. Feature engineering

События ограничиваются исходным 24-часовым окном. Генераторы считают признаки по событиям текущей cookie.


In [ ]:
clean, duplicate_stats, audit = antibot_pipeline.preprocess(train, test, events)
print(json.dumps(audit, indent=2))
(VALIDATION_DIR / "data_audit.json").write_text(json.dumps(audit, indent=2), encoding="utf-8")
assert audit["raw_events"] == 328905
assert audit["outside_events"] == 40779
assert audit["exact_duplicate_rows"] == 4293
assert audit["same_second_rows_preserved"] == 3020
assert audit["captcha_inside"] == 0
features = antibot_pipeline.make_features(clean, train, duplicate_stats)
test_features = antibot_pipeline.make_features(clean, test, duplicate_stats)
for name, x in features.items():
    print(name, x.shape, test_features[name].shape)
    assert list(x.columns) == list(test_features[name].columns)
    (VALIDATION_DIR / f"features_{name}.json").write_text(json.dumps(list(x.columns), indent=2), encoding="utf-8")
assert features["safe"].shape == (11091,202)
assert features["pointer"].shape == (11091,243)


## 4. Train models

Сохранены две LightGBM-модели, по 10 фиксированных seed на каждую. SAFE использует 202 признака; pointer-модель — те же признаки плюс 41 pointer-признак.


In [ ]:
print("SEEDS:", antibot_pipeline.SEEDS)
print("BASE_PARAMS:", antibot_pipeline.BASE_PARAMS)
print("POINTER_PARAMS:", antibot_pipeline.POINTER_PARAMS)


## 5. Validation

Temporal holdout зафиксирован в исходном решении: window_start_ts не раньше 2026-04-17. Validation обучает модели на более ранних датах.


In [ ]:
validation_metrics, validation_predictions = antibot_pipeline.validate(features, train, VALIDATION_DIR)
print(validation_metrics)
expected = 0.8484848484848485
if abs(validation_metrics["par"] - expected) > 1e-12:
    print("NOTICE: score отличается. Сравните версии зависимостей и исходные данные.")


## 6. Ensemble

Компоненты переводятся в ранги и объединяются с равными весами 50/50; внутри каждого компонента усредняются 10 seed.


In [ ]:
print("Validation P@R>=0.70:", validation_metrics["par"])
print("Отчёт по seed:", VALIDATION_DIR / "robustness_summary.csv")


## 7. Threshold selection

В исходном коде отдельный числовой cutoff не подбирается. Метрика берёт максимальный precision среди порогов с recall не ниже 0.70. Submission сохраняет непрерывный ensemble score.


## 8. Create submission

Модели обучаются на всём train. Submission сохраняет cookie_id в исходном порядке test.


In [ ]:
submission = antibot_pipeline.fit_full(features, test_features, train, test, OUTPUT_DIR)
assert len(submission) == 4909
assert list(submission.columns) == ["cookie_id", "score"]
assert submission.cookie_id.equals(test.cookie_id)
submission_path = OUTPUT_DIR / "submission.csv"
print(submission.head().to_string(index=False))
print("Saved:", submission_path.resolve())
try:
    from google.colab import files
    files.download(str(submission_path))
except ImportError:
    pass


## Output files

outputs/submission.csv — итог. outputs/validation/ содержит аудит данных, список признаков, метрики, устойчивость по seed и validation predictions. Кэши признаков не сохраняются.
